# TLog CLI Workflow

This notebook shows the command-line workflow for TLog. TLog is just another parser/compiler syntax: the generic `typedlogic convert`, `typedlogic dump`, and `typedlogic solve` commands do the work.

In [ ]:
from pathlib import Path

workspace = Path("tlog-demo")
workspace.mkdir(exist_ok=True)
ancestor_tlog = workspace / "ancestor.tlog"
ancestor_tlog.write_text(
    """
type PersonID: str.

pred parent(parent: PersonID, child: PersonID).
pred ancestor(ancestor: PersonID, descendant: PersonID).

parent("Alice", "Bob").
parent("Bob", "Charlie").

/// Direct parent links are ancestor links.
ancestor(x, y) :- parent(x, y).

/// Ancestor links are transitive.
ancestor(x, z) :- ancestor(x, y), ancestor(y, z).
    """.strip()
    + "\n",
    encoding="utf-8",
)
ancestor_tlog

## Convert TLog To Other Formats

The input format is inferred from `.tlog`; the output format is selected with `-t`.

In [ ]:
!typedlogic convert tlog-demo/ancestor.tlog -t prolog

In [ ]:
!typedlogic convert tlog-demo/ancestor.tlog -t yaml

## Convert Back To TLog

`tlog` is also a compiler target.

In [ ]:
!typedlogic convert tlog-demo/ancestor.tlog -t tlog

## Run Inference With Clingo

`--show` filters materialized ground terms by predicate. `--max-models` limits model output.

In [ ]:
!typedlogic solve tlog-demo/ancestor.tlog --solver clingo --show ancestor --max-models 1

## Literate Markdown

A `.tlog.md` file can mix prose and fenced TLog blocks.

In [ ]:
literate = workspace / "literate-rules.tlog.md"
literate.write_text(
    """
# Family Rules

Only fenced `tlog` blocks are parsed.

```tlog
pred parent(parent: str, child: str).
pred ancestor(ancestor: str, descendant: str).
parent("Alice", "Bob").
∀ x, y | parent(x, y) -> ancestor(x, y).
```
    """.strip()
    + "\n",
    encoding="utf-8",
)
literate

In [ ]:
!typedlogic dump tlog-demo/literate-rules.tlog.md -t prolog

## Multiple Worlds

When the selected solver supports multiple models, the same generic `solve` command can show them.

In [ ]:
worlds_tlog = workspace / "worlds.tlog"
worlds_tlog.write_text(
    """
pred selected(option: str).
pred hidden(option: str).

selected("tea") | selected("coffee").
hidden("noise").
    """.strip()
    + "\n",
    encoding="utf-8",
)
worlds_tlog

In [ ]:
!typedlogic solve tlog-demo/worlds.tlog --solver clingo --show selected --max-models 2